# Preparación de datos — House Prices

Este notebook reproduce de forma separada la **Etapa 3. Preparación de los datos**.  
Su salida son dos archivos:

- `train_clean.csv`
- `test_clean.csv`



In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train original:", train.shape)
print("Test original:", test.shape)


Train original: (1460, 81)
Test original: (1459, 80)


## 3.1. Seleccionar los datos

- `SalePrice` es la variable objetivo y solo existe en `train.csv`.
- `Id` se conserva como identificador para rastrear observaciones y construir el submission, pero **no se utilizará como predictor**.
- Se eliminan `GarageArea` y `TotRmsAbvGrd` del conjunto de predictores, de acuerdo con la decisión documentada de reducir redundancia por multicolinealidad.


In [2]:
# Variables descartadas por la decisión documentada de multicolinealidad
columnas_descartadas = ["GarageArea", "TotRmsAbvGrd"]

train = train.drop(columns=columnas_descartadas)
test = test.drop(columns=columnas_descartadas)

print("Train después de selección:", train.shape)
print("Test después de selección:", test.shape)


Train después de selección: (1460, 79)
Test después de selección: (1459, 78)


## 3.2. Limpiar los datos

La limpieza incluye:

1. Retirar del **train** los dos casos con `GrLivArea > 4000` y precio anormalmente bajo.
2. Corregir en el `test` oficial el valor imposible `GarageYrBlt = 2207`, sustituyéndolo por `YearBuilt`.
3. Tratar los NA que significan **ausencia de una característica** con la etiqueta `SinCaracteristica` o con `0`, según corresponda.
4. Imputar `LotFrontage` con la mediana por `Neighborhood`.
5. Para cualquier faltante restante, utilizar moda en variables categóricas y mediana en variables numéricas, calculadas a partir del train.


In [3]:
# 1) Outliers: solo se retiran del train, nunca del test oficial
mask_outliers = (
    (train["GrLivArea"] > 4000) &
    (train["SalePrice"] < 200000)
)

print("Outliers eliminados:", int(mask_outliers.sum()))
train = train.loc[~mask_outliers].copy()

# 2) Corrección del valor imposible 2207 en GarageYrBlt del test
mask_2207 = test["GarageYrBlt"] == 2207
print("Valores GarageYrBlt=2207 corregidos:", int(mask_2207.sum()))

test.loc[mask_2207, "GarageYrBlt"] = (
    test.loc[mask_2207, "YearBuilt"]
)


Outliers eliminados: 2
Valores GarageYrBlt=2207 corregidos: 1


In [4]:
# 3) NA que indican ausencia real de la característica
categoricas_ausencia = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2", "MasVnrType"
]

numericas_ausencia = [
    "GarageYrBlt", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
]

for col in categoricas_ausencia:
    if col in train.columns:
        train[col] = train[col].fillna("SinCaracteristica")
        test[col] = test[col].fillna("SinCaracteristica")

for col in numericas_ausencia:
    if col in train.columns:
        train[col] = train[col].fillna(0)
        test[col] = test[col].fillna(0)


In [5]:
# 4) LotFrontage: mediana por Neighborhood aprendida únicamente del train
mediana_frente_barrio = (
    train.groupby("Neighborhood")["LotFrontage"].median()
)
mediana_frente_global = train["LotFrontage"].median()

train["LotFrontage"] = (
    train["LotFrontage"]
    .fillna(train["Neighborhood"].map(mediana_frente_barrio))
    .fillna(mediana_frente_global)
)

test["LotFrontage"] = (
    test["LotFrontage"]
    .fillna(test["Neighborhood"].map(mediana_frente_barrio))
    .fillna(mediana_frente_global)
)


In [6]:
# 5) Faltantes restantes:
#    - categóricos -> moda del train
#    - numéricos   -> mediana del train

columnas_predictoras = [
    c for c in train.columns if c != "SalePrice"
]

for col in columnas_predictoras:
    if train[col].isna().any() or test[col].isna().any():

        if train[col].dtype == "object":
            moda = train[col].mode(dropna=True)
            valor_relleno = (
                moda.iloc[0] if not moda.empty else "SinDato"
            )
        else:
            valor_relleno = train[col].median()

        train[col] = train[col].fillna(valor_relleno)
        test[col] = test[col].fillna(valor_relleno)

print("NaN restantes en train:", train.isna().sum().sum())
print("NaN restantes en test:", test.isna().sum().sum())


NaN restantes en train: 0
NaN restantes en test: 0


## 3.3. Construir los datos

Se crean exactamente las cuatro variables nuevas descritas en el documento:

- `Edad_Casa`
- `Area_Total_Construida`
- `Total_Banos`
- `Es_Remodelada`


In [7]:
for data in [train, test]:
    data["Edad_Casa"] = (
        data["YrSold"] - data["YearBuilt"]
    ).clip(lower=0)

    data["Area_Total_Construida"] = (
        data["TotalBsmtSF"] +
        data["1stFlrSF"] +
        data["2ndFlrSF"]
    )

    data["Total_Banos"] = (
        data["FullBath"] +
        0.5 * data["HalfBath"] +
        data["BsmtFullBath"] +
        0.5 * data["BsmtHalfBath"]
    )

    data["Es_Remodelada"] = (
        data["YearRemodAdd"] != data["YearBuilt"]
    ).astype(int)

train[[
    "Edad_Casa",
    "Area_Total_Construida",
    "Total_Banos",
    "Es_Remodelada"
]].head()


,Edad_Casa,Area_Total_Construida,Total_Banos,Es_Remodelada
0,5,2566,3.5,0
1,31,2524,2.5,0
2,7,2706,3.5,1
3,91,2473,2.0,1
4,8,3343,3.5,0


## 3.4. Integrar los datos

No se integran múltiples fuentes. `train.csv` y `test.csv` pertenecen a la misma competencia y comparten la misma estructura de predictores.

El `train` se utiliza para aprender; el `test` oficial se reserva para generar predicciones al final.


## 3.5. Formatear los datos

- `MSSubClass` y `MoSold` se guardan como variables categóricas.
- Se crea `Log_SalePrice = log1p(SalePrice)` en el train.
- El **One-Hot Encoding** y el **StandardScaler** se aplicarán dentro del `Pipeline` del notebook de modelado.

Esta última decisión es deliberada: si se ajustara el escalador o el codificador usando todo el dataset antes de separar entrenamiento y prueba, habría riesgo de *data leakage*. Por eso aquí se deja el dataset limpio y el formateo aprendido se ajusta únicamente con los datos de entrenamiento.


In [8]:
for data in [train, test]:
    data["MSSubClass"] = data["MSSubClass"].astype(str)
    data["MoSold"] = data["MoSold"].astype(str)

train["Log_SalePrice"] = np.log1p(train["SalePrice"])

print("Train limpio:", train.shape)
print("Test limpio:", test.shape)
print("NaN train:", train.isna().sum().sum())
print("NaN test:", test.isna().sum().sum())

train.to_csv("train_clean.csv", index=False)
test.to_csv("test_clean.csv", index=False)

print("\nArchivos generados:")
print("- train_clean.csv")
print("- test_clean.csv")


Train limpio: (1458, 84)
Test limpio: (1459, 82)
NaN train: 0
NaN test: 0

Archivos generados:
- train_clean.csv
- test_clean.csv
